In [ ]:
import pandas as pd
import plotly.graph_objects as go
import pdfplumber
import requests

In [3]:
# Determine which districts to compare
target_districts = [
    'East Maine SD 63',
    'Winnetka SD 36',
    'Northbrook SD 28',
    'Glencoe SD 35',
    'CCSD 62',
    'Park Ridge CCSD 64',
    'Lincolnwood SD 74',
    'Arlington Heights SD 25',
    'Skokie SD 68',
    'Skokie SD 69',
    'Skokie SD 73-5',
    'Oak Park ESD 97',
    'Northbrook/Glenview SD 30',
    'Glenview CCSD 34',
    'Wilmette SD 39',
    'Wheeling CCSD 21',
    'Palatine CCSD 15',
    'North Shore SD 112',
    'Evanston CCSD 65'
]

In [57]:
# Define mapping from PositionCodeDescription or Position to Role
role_map = {
    'administrator in a bilingual education program': 'Admin',
    'administrator in  a bilingual education program': 'Admin',
    'assistant principal': 'Principal',
    'assistant special education director': 'Admin',
    'assistant/associate district superintendent': 'Admin',
    'bilingual education teacher': 'Teacher',
    'bilingual special education teacher': 'Teacher',
    'career and technical educator (cte)': 'Admin',
    'chief executive officer': 'Admin',
    'chief school business official': 'Admin',
    'citywide administrator': 'Admin',
    'citywide resource teacher': 'Teacher',
    'dean of students admin (admin endorsement held)': 'Admin',
    'dean of students teacher no admin endorsement)': 'Admin',
    'director area voc cent or supervisor or more 1 field in cte': 'Admin',
    'district superintendent': 'Admin',
    'english as a second language teacher': 'Teacher',
    'general administrator or general supervisor': 'Admin',
    'head teacher': 'Teacher',
    'head of gen ed (depart chair admin endorsement held)': 'Admin',
    'head of gen ed (department chair no admin endorsement held)': 'Admin',
    'principal': 'Principal',
    'reading teacher': 'Teacher',
    'resource teacher arts(visual art, music, drama, and theatre)': 'Teacher',
    'resource teacher economics': 'Teacher',
    'resource teacher elementary': 'Teacher',
    'resource teacher english/language arts': 'Teacher',
    'resource teacher foreign language': 'Teacher',
    'resource teacher government/civics/political science': 'Teacher',
    'resource teacher history': 'Teacher',
    'resource teacher math': 'Teacher',
    'resource teacher other': 'Teacher',
    'resource teacher reading': 'Teacher',
    'resource teacher science (all sciences)': 'Teacher',
    'special education director': 'Admin',
    'special education supervisor': 'Admin',
    'special education teacher': 'Teacher',
    'speech language pathology teacher': 'Teacher',
    'supervisor of more than one school support personnel area': 'Admin',
    'supervisor of one field in career and technical education': 'Admin',
    'supervisor of one school support personnel area': 'Admin',
    'supervisory dean': 'Admin',
    'teacher': 'Teacher',
    'visiting international teacher': 'Teacher',
    '1st grade teacher': 'Teacher',
    '2nd grade teacher': 'Teacher',
    '3rd grade teacher': 'Teacher',
    '4th grade teacher': 'Teacher',
    '5th grade teacher': 'Teacher',
    'academic instructional coach': 'Teacher',
    'accelerated ela coordinator': 'Admin',
    'african centered curr 2nd grade t': 'Teacher',
    'african centered curr 3rd grade t': 'Teacher',
    'african centered curr 4th grade t': 'Teacher',
    'african centered curr 5th grade t': 'Teacher',
    'african centered curr k grade tea': 'Teacher',
    'art / media art teacher': 'Teacher',
    'art teacher': 'Teacher',
    'assistant superintendent of acade': 'Admin',
    'assistant superintendent of academics': 'Admin',
    'assistant superintendent of acco': 'Admin',
    'assistant superintendent of accountability': 'Admin',
    'asst director of teaching & learni': 'Admin',
    'asst director of teaching & learning': 'Admin',
    'band teacher': 'Teacher',
    'bilingual arts/reading teacher': 'Teacher',
    'bilingual esl teacher': 'Teacher',
    'bilingual math/science teacher': 'Teacher',
    'bilingual pre-k hs teacher': 'Teacher',
    'bilingual pre-k pfa teacher': 'Teacher',
    'bilingual teacher': 'Teacher',
    'chief financial officer': 'Admin',
    'computer science teacher': 'Teacher',
    'dec president': 'Admin',
    'director of buildings, grounds & t': 'Admin',
    'director of climate & safety': 'Admin',
    'director of early childhood prog': 'Admin',
    'director of early childhood programs': 'Admin',
    'director of finance': 'Admin',
    'director of human relations': 'Admin',
    'director of humanities': 'Admin',
    'director of mtss & sel': 'Admin',
    'director of multilingual services': 'Admin',
    'director of programs & partnersh': 'Admin',
    'director of programs & partnerships': 'Admin',
    'director of schools management': 'Admin',
    'director of steam': 'Admin',
    'director of strategic projects': 'Admin',
    'director of student specialized se': 'Admin',
    'director of student specialized services': 'Admin',
    'drama teacher': 'Teacher',
    'drama/dance teacher': 'Teacher',
    'esl teacher': 'Teacher',
    'executive chief of communications': 'Admin',
    'executive chief of human relations': 'Admin',
    'executive director of raad': 'Admin',
    'executive director of technology': 'Admin',
    'french teacher': 'Teacher',
    'health services director': 'Admin',
    'hearing impaired teacher': 'Teacher',
    'hearing itinerant': 'Teacher',
    'iep interventionist': 'Teacher',
    'ies coordinator': 'Admin',
    'ies educator': 'Teacher',
    'ies educator - adaptive pe': 'Teacher',
    'ies educator bilingual': 'Teacher',
    'interventionist': 'Teacher',
    'interventionist bilingual': 'Teacher',
    'kindergarten teacher': 'Teacher',
    'language arts teacher': 'Teacher',
    'language arts/literature teacher': 'Teacher',
    'language arts/ss teacher': 'Teacher',
    'leave of absence': 'LOA',
    'library media specialist': 'Teacher',
    'manager of stud specialized servic': 'Admin',
    'math and science teacher': 'Teacher',
    'math teacher': 'Teacher',
    'media arts teacher': 'Teacher',
    'multilingual coordinator': 'Admin',
    'music teacher': 'Teacher',
    'occupational therapist': 'Teacher',
    'orchestra teacher': 'Teacher',
    'physical education teacher': 'Teacher',
    'physical therapist': 'Teacher',
    'pre-k hs teacher': 'Teacher',
    'pre-k pfa teacher': 'Teacher',
    'psychologist': 'Teacher',
    'school counselor': 'Teacher',
    'school nurse': 'Teacher',
    'science teacher': 'Teacher',
    'science/social studies teacher': 'Teacher',
    'social studies teacher': 'Teacher',
    'social worker': 'Teacher',
    'spanish & french teacher': 'Teacher',
    'spanish teacher': 'Teacher',
    'speech language pathologist': 'Teacher',
    'superintendent': 'Admin',
    'teacher visually impaired': 'Teacher',
    'twi 1st grade teacher': 'Teacher',
    'twi 2nd grade teacher': 'Teacher',
    'twi 3rd grade teacher': 'Teacher',
    'twi 4th grade teacher': 'Teacher',
    'twi 5th grade teacher': 'Teacher',
    'twi kindergarten teacher': 'Teacher',
    'vocational teacher': 'Teacher'
}

In [59]:
# Import the 2024 EIS Salary Data Set from https://www.isbe.net/pages/educator-employment-information.aspx
salary_24 = pd.read_excel('https://www.isbe.net/_layouts/Download.aspx?SourceUrl=/Documents/2024-ATSB-Report.xlsx')

In [116]:
# Rename EntityName to District
salary_24 = salary_24.rename(columns={'EntityName': 'District'})

# Filter salary data to only include target districts
salary_24 = salary_24[salary_24['District'].isin(target_districts)].copy()

# Remove duplicate entries for the same person within the same district
salary_24 = salary_24.drop_duplicates(subset=['LastName', 'FirstName', 'District']).copy()

# Add TotalComp column
salary_24['TotalComp'] = salary_24[[
    'BaseSalary', 'RetirementEnhancements', 'OtherBenefits']].sum(axis=1)

# Create new column 'Role' by mapping the PositionCodeDescription column
salary_24['Role'] = salary_24['PositionCodeDescription'].str.lower().map(role_map)

In [ ]:
# Import the D65 2025-2026 Administrator and Teacher Salary Schedule PDF

url = "https://resources.finalsite.net/images/v1759244219/district65net/uo4v3f631tarj7khyawc/50PA97-0256AdministratorandTeacherFY26.pdf"

# Download the PDF
response = requests.get(url)
with open('temp.pdf', 'wb') as f:
    f.write(response.content)

# Extract tables from all pages
all_tables = []

with pdfplumber.open('temp.pdf') as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        if table:  # Check if table exists on this page
            all_tables.append(table)

# Convert to DataFrames and concatenate
# First table has headers
df_list = []
headers = all_tables[0][0]  # Get headers from first page

for i, table in enumerate(all_tables):
    if i == 0:
        # First page: include headers
        df_list.append(pd.DataFrame(table[1:], columns=headers))
    else:
        # Subsequent pages: skip header row if it repeats
        # Check if first row matches headers
        if table[0] == headers:
            df_list.append(pd.DataFrame(table[1:], columns=headers))
        else:
            df_list.append(pd.DataFrame(table, columns=headers))

# Concatenate all dataframes
d65_25_salaries = pd.concat(df_list, ignore_index=True)


In [118]:
# List of columns to clean and convert
salary_columns = ['Total Salary', 'Dental\nInsurance', 'Health\nInsurance', 
                  'Vision\nInsurance', 'Life\nInsurance']

# Apply transformation to each column
for col in salary_columns:
    d65_25_salaries[col] = d65_25_salaries[col].astype(str).str.replace(r'[^\d.]', '', regex=True)
    d65_25_salaries[col] = pd.to_numeric(d65_25_salaries[col], errors='coerce')

# Calculate TotalComp
d65_25_salaries['TotalComp'] = d65_25_salaries[[
    'Total Salary',
    'Dental\nInsurance',
    'Health\nInsurance',
    'Vision\nInsurance',
    'Life\nInsurance'
]].sum(axis=1)

# Rename columns for consistency
d65_25_salaries = d65_25_salaries.rename(columns={
    'Last Name': 'LastName',
    'First Name': 'FirstName'
})
# Remove duplicate entries for the same person within the same district
d65_25_salaries = d65_25_salaries.drop_duplicates(subset=['LastName', 'FirstName']).copy()

d65_25_salaries['District'] = 'Evanston CCSD 65 - 2025'

# Create new column 'Role' by mapping the PositionCodeDescription column
d65_25_salaries['Role'] = d65_25_salaries['Position'].str.lower().map(role_map)

In [119]:
# Add the D65 2025 data to salary_24
combined_salary_data = pd.concat([salary_24, d65_25_salaries], ignore_index=True)

In [120]:
# Aggregate salary data by district and role
salary_summary = combined_salary_data.groupby(['District', 'Role']).agg(
    salary_total=('TotalComp', 'sum'),
    headcount=('TotalComp', 'count')
).reset_index()

salary_summary['salary_total'] = salary_summary['salary_total'].round(2)

salary_summary['avg_salary'] = (salary_summary['salary_total'] / salary_summary['headcount']).round(2)

In [80]:
# Add enrollment data for each district
# Download the 2024 Report Card Public Data Set from https://www.isbe.net/pages/report-card.aspx
report_card_24 = pd.read_excel("https://www.isbe.net/_layouts/Download.aspx?SourceUrl=/Documents/24-RC-Pub-Data-Set.xlsx", sheet_name='General')

In [121]:
# Filter report card data to only include target districts
filtered_report_card = report_card_24[report_card_24['District'].isin(target_districts)].copy()

# Filter to only District data
enroll = filtered_report_card[filtered_report_card['Type'] == 'District'].copy()

enroll = enroll.rename(columns={'# Student Enrollment': 'enrollment'})

In [122]:
# Count number of schools per district
school_counts = filtered_report_card[filtered_report_card['Type'] == 'School'].groupby('District').size().reset_index(name='num_schools')

In [124]:
# Add total district enrollments to salary summary
salary_summary = salary_summary.merge(
    enroll[['District', 'enrollment']], 
    left_on='District', 
    right_on='District', 
    how='left'
)

In [125]:
# Add school counts to salary summary
salary_summary = salary_summary.merge(
    school_counts,
    left_on='District',
    right_on='District',
    how='left'
)

In [126]:
# Set D65 2025 enrollment and number of schools
salary_summary.loc[salary_summary['District'] == 'Evanston CCSD 65 - 2025', 'enrollment'] = 5922  # 2025 enrollment number from https://resources.finalsite.net/images/v1746134088/district65net/ujp5dsbfsk5xdl9qzujo/TotalBuildingSFwFoster.pdf
salary_summary.loc[salary_summary['District'] == 'Evanston CCSD 65 - 2025', 'num_schools'] = 16 # No change from 2024-2025 school year

In [127]:
# Add total salary per 1000 students and headcount per 1000 students
salary_summary['salary_per_1000_students'] = (salary_summary['salary_total'] / salary_summary['enrollment'] * 1000).round(2)
salary_summary['headcount_per_1000_students'] = (salary_summary['headcount'] / salary_summary['enrollment'] * 1000).round(2)

# Add total salary per school and headcount per school
salary_summary['salary_per_school'] = (salary_summary['salary_total'] / salary_summary['num_schools']).round(2)
salary_summary['headcount_per_school'] = (salary_summary['headcount'] / salary_summary['num_schools']).round(2) 

# Visualizations

In [128]:
# Enrollment by district
enroll_by_district = salary_summary[salary_summary['Role'] == 'Admin'].sort_values('enrollment', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in enroll_by_district['District']]

# Plot Enrollment by District
fig = go.Figure()
fig.add_trace(go.Bar(
    x=enroll_by_district['enrollment'],
    y=y_labels,
    orientation='h',
    text=enroll_by_district['enrollment'],
    textposition='auto'
))
fig.update_layout(
    title='Student Enrollment by District',
    xaxis_title='Number of Students',
    yaxis_title='District',
    height=800
)
fig.show()


In [129]:
# School Count by district
school_num_by_district = salary_summary[salary_summary['Role'] == 'Admin'].sort_values('num_schools', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in school_num_by_district['District']]

# Plot Enrollment by District
fig = go.Figure()
fig.add_trace(go.Bar(
    x=school_num_by_district['num_schools'],
    y=y_labels,
    orientation='h',
    text=school_num_by_district['num_schools'],
    textposition='auto'
))
fig.update_layout(
    title='Number of Schools by District',
    xaxis_title='Number of Schools',
    yaxis_title='District',
    height=800
)
fig.show()


## Admin Data

In [130]:
# Plot Number of Admin Positions by District
admin_data = salary_summary[salary_summary['Role'] == 'Admin'].sort_values('headcount', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in admin_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=admin_data['headcount'],
    y=y_labels,
    orientation='h',
    text=admin_data['headcount'],
    textposition='auto'
))

fig.update_layout(
    title='Number of Admin Positions by District',
    xaxis_title='Headcount',
    yaxis_title='District',
    height=800
)

fig.show()

In [131]:
# Plot Number of Admin Positions per 1k Students by District
admin_data = admin_data.sort_values('headcount_per_1000_students', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in admin_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=admin_data['headcount_per_1000_students'],
    y=y_labels,
    orientation='h',
    text=admin_data['headcount_per_1000_students'],
    textposition='auto'    
))

fig.update_layout(
    title='Number of Admin Positions per 1k Students by District',
    xaxis_title='Headcount/1k Students',
    yaxis_title='District',
    height=800
)

fig.show()

In [132]:
# Plot Number of Admin Positions per Number of Schools in each District
admin_data = admin_data.sort_values('headcount_per_school', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in admin_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=admin_data['headcount_per_school'],
    y=y_labels,
    orientation='h',
    text=admin_data['headcount_per_school'],
    textposition='auto'    
))

fig.update_layout(
    title='Number of Admin Positions per School in Each District',
    xaxis_title='Headcount/School',
    yaxis_title='District',
    height=800
)

fig.show()

In [134]:
# Plot Total Salaries of Admin Positions per Number of Schools in each District
admin_data = admin_data.sort_values('salary_per_school', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in admin_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=admin_data['salary_per_school'],
    y=y_labels,
    orientation='h',
    text=admin_data['salary_per_school'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'    
))

fig.update_layout(
    title='Total Admin Salaries per School in Each District',
    xaxis_title='Total Salary/School',
    yaxis_title='District',
    height=800
)

fig.show()

In [135]:
# Plot Total Salaries of Admin Positions per 1k Students by District
admin_data = admin_data.sort_values('salary_per_1000_students', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in admin_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=admin_data['salary_per_1000_students'],
    y=y_labels,
    orientation='h',
    text=admin_data['salary_per_1000_students'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'
))

fig.update_layout(
    title='Total Salaries of Admin Positions per 1k Students by District',
    xaxis_title='Salaries/1k Students',
    yaxis_title='District',
    height=800
)

fig.show()

In [136]:
# Calculate extra Admin cost based on average headcount per 1k students across all districts multiplied by the admin salary per district
admin_avg_headcount_per_1000 = admin_data['headcount_per_1000_students'].mean()

# Extra admins per 1000 students times their avg salary times enrollment scale
admin_data['extra_admin_cost'] = (
    (admin_data['headcount_per_1000_students'] - admin_avg_headcount_per_1000) * 
    admin_data['avg_salary'] * 
    (admin_data['enrollment'] / 1000)
)
admin_data['extra_admin_cost'] = admin_data['extra_admin_cost'].round(2)
admin_data = admin_data.sort_values('extra_admin_cost', ascending=False)

# Plot Extra Admin Cost by District
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in admin_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=admin_data['extra_admin_cost'],
    y=y_labels,
    orientation='h',
    text=admin_data['extra_admin_cost'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'
))

fig.update_layout(
    title='Estimated Extra Admin Cost by District',
    xaxis_title='Extra Admin Cost ($)',
    yaxis_title='District',
    height=800,
    annotations=[
        dict(
            text="This is calculated by setting each district's admin headcount per 1,000 students to the average across all districts, then calculating the cost difference based on each district's average admin salary.",
            xref="paper", yref="paper",
            x=0.5, y=-0.1,
            showarrow=False,
            xanchor='center',
            font=dict(size=10, color="gray")
        )
    ]
)

fig.show()

## Principal Data

In [137]:
# Plot Headcount for Principals by District
principal_data = salary_summary[salary_summary['Role'] == 'Principal'].sort_values('headcount', ascending=False)
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in principal_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=principal_data['headcount'],
    y=y_labels,
    orientation='h',
    text=principal_data['headcount'],
    textposition='auto'
))
fig.update_layout(
    title='Number of Principal Positions by District',
    xaxis_title='Headcount',
    yaxis_title='District',
    height=800
)
fig.show()

In [138]:
# Plot Headcount per 1k Students for Principals by District
principal_data = principal_data.sort_values('headcount_per_1000_students', ascending=False)
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in principal_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=principal_data['headcount_per_1000_students'],
    y=y_labels,
    orientation='h',
    text=principal_data['headcount_per_1000_students'],
    textposition='auto'
))
fig.update_layout(
    title='Number of Principal Positions per 1k Students by District',
    xaxis_title='Headcount/1k Students',
    yaxis_title='District',
    height=800
)
fig.show()

In [139]:
# Plot Number of Principal Positions per Number of Schools in each District
principal_data = principal_data.sort_values('headcount_per_school', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in principal_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=principal_data['headcount_per_school'],
    y=y_labels,
    orientation='h',
    text=principal_data['headcount_per_school'],
    textposition='auto'    
))

fig.update_layout(
    title='Number of Principal Positions per School in Each District',
    xaxis_title='Headcount/School',
    yaxis_title='District',
    height=800
)

fig.show()

In [140]:
# Plot Total Salaries of Principal Positions per Number of Schools in each District
principal_data = principal_data.sort_values('salary_per_school', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in principal_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=principal_data['salary_per_school'],
    y=y_labels,
    orientation='h',
    text=principal_data['salary_per_school'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'    
))

fig.update_layout(
    title='Total Principal Salaries per School in Each District',
    xaxis_title='Total Salary/School',
    yaxis_title='District',
    height=800
)

fig.show()

In [141]:
# Plot Salary per 1k Students for Principals by District
principal_data = principal_data.sort_values('salary_per_1000_students', ascending=False)
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in principal_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=principal_data['salary_per_1000_students'],
    y=y_labels,
    orientation='h',
    text=principal_data['salary_per_1000_students'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'
))
fig.update_layout(
    title='Total Salaries of Principal Positions per 1k Students by District',
    xaxis_title='Salaries/1k Students',
    yaxis_title='District',
    height=800
)
fig.show()

In [142]:
# Calculate extra Principal cost based on average headcount per 1k students across all districts
principal_avg_headcount_per_1000 = principal_data['headcount_per_1000_students'].mean()
# Extra principals per 1000 students times their avg salary times enrollment scale
principal_data['extra_principal_cost'] = (
    (principal_data['headcount_per_1000_students'] - principal_avg_headcount_per_1000) * 
    principal_data['avg_salary'] * 
    (principal_data['enrollment'] / 1000)
)
principal_data['extra_principal_cost'] = principal_data['extra_principal_cost'].round(2)
principal_data = principal_data.sort_values('extra_principal_cost', ascending=False)

# Plot Extra Principal Cost by District
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in principal_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=principal_data['extra_principal_cost'],
    y=y_labels,
    orientation='h',
    text=principal_data['extra_principal_cost'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'
))
fig.update_layout(
    title='Estimated Extra Principal Cost by District',
    xaxis_title='Extra Principal Cost ($)',
    yaxis_title='District',
    height=800,
    annotations=[
        dict(
            text="This is calculated by setting each district's principal headcount per 1,000 students to the average across all districts, then calculating the cost difference based on each district's average principal salary.",
            xref="paper", yref="paper",
            x=0.5, y=-0.1,
            showarrow=False,
            xanchor='center',
            font=dict(size=10, color="gray")
        )
    ]
)
fig.show()

## Teacher Data

In [143]:
# Plot Headcount for Teachers by District
teacher_data = salary_summary[salary_summary['Role'] == 'Teacher'].sort_values('headcount', ascending=False)
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in teacher_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=teacher_data['headcount'],
    y=y_labels,
    orientation='h',
    text=teacher_data['headcount'],
    textposition='auto'
))
fig.update_layout(
    title='Number of Teacher Positions by District',
    xaxis_title='Headcount',
    yaxis_title='District',
    height=800
)
fig.show()

In [144]:
# Plot Headcount per 1k Students for Teachers by District
teacher_data = teacher_data.sort_values('headcount_per_1000_students', ascending=False)
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in teacher_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=teacher_data['headcount_per_1000_students'],
    y=y_labels,
    orientation='h',
    text=teacher_data['headcount_per_1000_students'],
    textposition='auto'
))
fig.update_layout(
    title='Number of Teacher Positions per 1k Students by District',
    xaxis_title='Headcount/1k Students',
    yaxis_title='District',
    height=800
)
fig.show()

In [145]:
# Plot Number of Teacher Positions per Number of Schools in each District
teacher_data = teacher_data.sort_values('headcount_per_school', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in teacher_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=teacher_data['headcount_per_school'],
    y=y_labels,
    orientation='h',
    text=teacher_data['headcount_per_school'],
    textposition='auto'    
))

fig.update_layout(
    title='Number of Teacher Positions per School in Each District',
    xaxis_title='Headcount/School',
    yaxis_title='District',
    height=800
)

fig.show()

In [146]:
# Plot Total Salaries of Teacher Positions per Number of Schools in each District
teacher_data = teacher_data.sort_values('salary_per_school', ascending=False)

# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in teacher_data['District']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=teacher_data['salary_per_school'],
    y=y_labels,
    orientation='h',
    text=teacher_data['salary_per_school'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'    
))

fig.update_layout(
    title='Total Teacher Salaries per School in Each District',
    xaxis_title='Total Salary/School',
    yaxis_title='District',
    height=800
)

fig.show()

In [147]:
# Plot Salary per 1k Students for Teachers by District
teacher_data = teacher_data.sort_values('salary_per_1000_students', ascending=False)
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in teacher_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=teacher_data['salary_per_1000_students'],
    y=y_labels,
    orientation='h',
    text=teacher_data['salary_per_1000_students'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'
))
fig.update_layout(
    title='Total Salaries of Teacher Positions per 1k Students by District',
    xaxis_title='Salaries/1k Students',
    yaxis_title='District',
    height=800
)
fig.show()

In [148]:
# Calculate extra Teacher cost based on average headcount per 1k students across all districts
teacher_avg_headcount_per_1000 = teacher_data['headcount_per_1000_students'].mean()
# Extra teachers per 1000 students times their avg salary times enrollment scale
teacher_data['extra_teacher_cost'] = (
    (teacher_data['headcount_per_1000_students'] - teacher_avg_headcount_per_1000) * 
    teacher_data['avg_salary'] * 
    (teacher_data['enrollment'] / 1000)
)
teacher_data['extra_teacher_cost'] = teacher_data['extra_teacher_cost'].round(2)
teacher_data = teacher_data.sort_values('extra_teacher_cost', ascending=False)

# Plot Extra Teacher Cost by District
# Make Evanston label bold
y_labels = ['<b>' + district + '</b>' if district == 'Evanston CCSD 65' or district == 'Evanston CCSD 65 - 2025' else district 
            for district in teacher_data['District']]
fig = go.Figure()
fig.add_trace(go.Bar(
    x=teacher_data['extra_teacher_cost'],
    y=y_labels,
    orientation='h',
    text=teacher_data['extra_teacher_cost'].apply(lambda x: f'${x:,.0f}'),
    textposition='auto'
))
fig.update_layout(
    title='Estimated Extra Teacher Cost by District',
    xaxis_title='Extra Teacher Cost ($)',
    yaxis_title='District',
    height=800,
    annotations=[
        dict(
            text="This is calculated by setting each district's teacher headcount per 1,000 students to the average across all districts, then calculating the cost difference based on each district's average teacher salary.",
            xref="paper", yref="paper",
            x=0.5, y=-0.1,
            showarrow=False,
            xanchor='center',
            font=dict(size=10, color="gray")
        )
    ]
)
fig.show()